# Ingest recent daily archives

This notebook fills the gap between the last completed MMSDM month and the current files. A daily DispatchIS archive contains another layer of five-minute ZIPs, so its extraction differs from the monthly and current sources.

### Storage prerequisite

Before running the pipeline, I created the Databricks Volume `/Volumes/workspace/default/aemo_mlops_volume`. The ingestion notebooks write their Bronze files there, and the Silver transformation reads from the same location.

## 1. Discover recent daily archives

The archive page lists one file per day. The notebook parses those filenames and keeps the latest three months currently used by the pipeline.

### How the daily filename pattern works

```python
r"PUBLIC_DISPATCHIS_\d{8}\.zip"
```

- `r"..."` makes this a raw Python string, so backslashes are passed directly to the pattern matcher.
- `PUBLIC_DISPATCHIS_` requires the fixed prefix used by daily DispatchIS archives.
- `\d{8}` requires exactly eight digits: the archive date in `YYYYMMDD` form.
- `\.zip` requires the literal `.zip` extension.

For example, the pattern accepts `PUBLIC_DISPATCHIS_20260809.zip`. The later `r"\d{8}"` search extracts `20260809` from that filename so the notebook can keep only recent dates.

In [0]:
import os
import re
import time
import zipfile
from datetime import datetime

import requests
from dateutil.relativedelta import relativedelta


base_url = (
    "https://www.nemweb.com.au/"
    "REPORTS/ARCHIVE/DispatchIS_Reports/"
)

# Read the HTML listing because a new dated archive is added each day.
html = requests.get(base_url).text

# Daily archive names contain one YYYYMMDD date.
files = re.findall(
    r"PUBLIC_DISPATCHIS_\d{8}\.zip",
    html
)

# Limit this layer to the recent period that follows the monthly archive.
start_date = datetime.today() - relativedelta(months=3)

files = [
    file
    for file in sorted(set(files))
    if datetime.strptime(
        re.search(r"\d{8}", file).group(),
        "%Y%m%d"
    ) >= start_date
]

urls = [
    base_url + file
    for file in files
]

print(f"Found {len(urls)} files")

In [0]:
# Display the selected daily URLs before downloading them.
urls

## 2. Download only missing daily ZIPs

The original daily archives are retained in Bronze. Skipping existing files makes the ingestion safe to run again.

In [0]:
def download_if_not_exists(url, bronze_folder):
    filename = url.split("/")[-1]
    path = f"{bronze_folder}/{filename}"

    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")

    start = time.time()

    data = requests.get(url).content

    # Store the original daily archive before extracting it.
    with open(path, "wb") as file:
        file.write(data)

    print(f"Finished: {filename} - {time.time() - start:.1f} seconds")

    return path

In [0]:
bronze_folder = "/Volumes/workspace/default/aemo_mlops_volume/bronze/daily"

os.makedirs(bronze_folder, exist_ok=True)

for url in urls:
    download_if_not_exists(url, bronze_folder)

## 3. Unpack both archive levels

The outer file represents a day. Its contents are five-minute ZIPs, and those inner ZIPs contain the CSV files required by Silver.

In [0]:
csv_folder = bronze_folder + "_uncompressed"

os.makedirs(csv_folder, exist_ok=True)

zip_files = [
    file for file in os.listdir(bronze_folder)
    if file.endswith(".zip")
]

start = time.time()

print(f"Found {len(zip_files)} daily ZIP files\n")

# First level: one daily ZIP expands into many five-minute ZIPs.
for i, filename in enumerate(sorted(zip_files), 1):

    path = os.path.join(bronze_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:

        files = zip_file.namelist()

        already_extracted = all(
            os.path.exists(os.path.join(csv_folder, file))
            for file in files
        )

        if already_extracted:
            print(f"[{i}/{len(zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(zip_files)}] Extracting: {filename}")

        zip_file.extractall(csv_folder)


# Second level: each five-minute ZIP expands into its DispatchIS CSV.
inner_zip_files = [
    file for file in os.listdir(csv_folder)
    if file.endswith(".zip")
]

print(f"\nFound {len(inner_zip_files)} inner ZIP files\n")

for i, filename in enumerate(sorted(inner_zip_files), 1):

    path = os.path.join(csv_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:
        zip_file.extractall(csv_folder)

    # The inner ZIP is no longer needed after its CSV has been extracted.
    os.remove(path)

    print(f"[{i}/{len(inner_zip_files)}] Extracted: {filename}")


print()
print(f"Finished in {time.time() - start:.1f} seconds")